In [ ]:
import sys
sys.path.append("../eng-quant/")
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [26]:
from generators.toy_problem_to_json import toy_problem_to_json # type: ignore[reportMissingImports]
from relaxers.quadratic_model_lin_approx import quadratic_model_lin_approx # type: ignore[reportMissingImports]
from relaxers.quadratic_lin_approx_no_surrogate import quadratic_lin_approx_no_surrogate # type: ignore[reportMissingImports]
from relaxers.quadratic_lin_approx_nosur_refine import quadratic_lin_approx_no_surrogate_refine # type: ignore[reportMissingImports]
from testers.classical_solve_lin_program import solve_lp_return_x # type: ignore[reportMissingImports]
from testers.classical_solve_quad_program import solve_qcp_return_x # type: ignore[reportMissingImports]
from testers.condition_number_lin_program import condition_number_nes_basic # type: ignore[reportMissingImports]

import matplotlib.pyplot as plt
import numpy as np
import bisect

In [ ]:
# Number of copies of the network in the toy system
N=10

# Whether the production and flow variables have upper bounds
F_UPPER_BOUNDS=True

# Whether the demand constraint is an equality or inequality
DEMAND_INEQUALITY=True

# Capacity / demand parameters
# GAMMA=1.0
GAMMA=1.0
LAMBD=1.0
EPS=0.1
DELTA=0.1

# Lower bounds and upper bound on pressure
LX=LP=LF=0.0
UP=0.5

LIN_APPROX = 2

# Whether to do an inner or outer approximation
OUTER_APPROXIMATION = False

# Whether the coefficient is part of the surrogate
COEFFICIENT_SURROGATE = True

# Whether to bound the surrogates with bound constraints.
SURROGATE_BOUND_BELOW = True
SURROGATE_BOUND_ABOVE = True

# Whether to divide by lambda in an inner approximation
REMOVE_DIVISION = False

In [28]:
toy_problem_to_json(N, F_UPPER_BOUNDS, DEMAND_INEQUALITY, GAMMA, LAMBD,
                        EPS, DELTA, LP, LF, UP)
quad, q_time, q_x = solve_qcp_return_x(f_name="toy.json")
print(f"Exact objective value: {quad}")
print(f"Exact solution: {q_x}")

Gurobi Optimizer version 13.0.2 build v13.0.2rc1 (win64 - Windows 11+.0 (26300.2))

CPU model: AMD Ryzen 7 8845HS w/ Radeon 780M Graphics, instruction set [SSE2|AVX|AVX2|AVX512]
Thread count: 8 physical cores, 16 logical processors, using up to 16 threads

Presolve a model with 0 rows, 99 columns and 0 nonzeros (Min)
Model fingerprint: 0xac0f4106
Model has 30 linear objective coefficients
Model has 69 quadratic constraints
Coefficient statistics:
  Matrix range     [0e+00, 0e+00]
  QMatrix range    [1e+00, 1e+00]
  QLMatrix range   [1e+00, 1e+00]
  Objective range  [9e-01, 1e+00]
  Bounds range     [5e-01, 1e+00]
  RHS range        [0e+00, 0e+00]
  QRHS range       [1e+00, 1e+00]
Presolve removed 0 rows and 30 columns
Statistics for model 'qcp_from_json_pre':
  Problem type                : QCP
  Linear constraint matrix    : 167 rows, 167 columns, 332 nonzeros
  Variable types              : 167 continuous, 0 integer (0 binary)
  Quadratic constraints       : 49 constrs, 147 nonzeros


In [29]:
quadratic_lin_approx_no_surrogate(1, OUTER_APPROXIMATION, remove_division=REMOVE_DIVISION,
                            f_name="toy.json", endpoints=False)
val, time, x = solve_lp_return_x()
print(x)

# print(condition_number_nes_basic())
print(f"val: {val}")
print(f"Error: {val - quad}")

[0.4999999999985724, 0.4999999999540739, 0.3119683719668064, 0.4999999999984838, 0.4999999999866611, 0.308858122938345, 0.4999999999984333, 0.49999999998070144, 0.30785670953351996, 0.4999999999984179, 0.4999999999808094, 0.30748189309280644, 0.4999999999984249, 0.499999999979669, 0.307286414604749, 0.4999999999984484, 0.49999999997792977, 0.30711357091638086, 0.49999999999849076, 0.4999999999793571, 0.3068111469210612, 0.49999999999855715, 0.49999999997848055, 0.3061472804199178, 0.4999999999986086, 0.4999999999347565, 0.3048152859166064, 0.49999999999886735, 0.49999999962948666, 0.30346705135983143, 0.35450105308589014, 0.3677898010675643, 0.3716204888799281, 0.3728335995426673, 0.37332975297227394, 0.37372248200449115, 0.3744530842949373, 0.37629870366269613, 0.3809135265055314, 0.38800025048149683, 1.3575801184129536e-12, 2.0567255761253543e-12, 2.1960911219592033e-12, 2.212708633740638e-12, 2.178536225427179e-12, 2.102386184352713e-12, 1.965627064810941e-12, 1.6428215223466545e-12

In [ ]:
# Define a set of functions for each


quads = [[val] for val in x[(5 * N):]]
# print(quads)
iters = 5

for i in range(1, iters + 1):
    print(f"Iteration {i}")
    points_functions = []

    def np_uniform_add_q_factory(qs):
        def np_uniform_add_qs(lower, upper, num):
            refinement_points = len(qs)
            uniform = np.linspace(lower, upper, num - refinement_points)
            for q in qs:
                idx = np.searchsorted(uniform, q)
                uniform = np.insert(uniform, idx, q)
            return uniform
        return np_uniform_add_qs

    for qs in quads:
        points_functions.append(np_uniform_add_q_factory(qs))

    quadratic_lin_approx_no_surrogate_refine(LIN_APPROX + i, OUTER_APPROXIMATION, points_function=points_functions,
                                                remove_division=REMOVE_DIVISION, f_name="toy.json", endpoints=False)

    val, time, x = solve_lp_return_x()

    # print(f"x: {x}")
    print(f"val: {val}")
    print(f"Error: {val - quad}")

    for j, qs in enumerate(quads):
        qs.append(x[(5 * N) + j])

    # print(quads)

print(x)
print(f"Error: {val - quad}")

Iteration 1
val: 11.105126905362905
Error: 0.005125383695018471
Iteration 2
val: 11.10007678965474
Error: 7.52679868529782e-05
Iteration 3
val: 11.100001150689634
Error: -3.709782525618266e-07
Iteration 4
val: 11.100000018024383
Error: -1.50364350304244e-06
Iteration 5
val: 11.100000316746758
Error: -1.2049211282061378e-06
[0.4999999967636656, 0.4225283085612806, 0.48147515960121257, 0.49999999684613067, 0.4296605647650111, 0.48467934798569995, 0.49999999683259766, 0.4298458989056193, 0.48565648935918987, 0.49999999682470503, 0.43005978865530137, 0.48676391134642605, 0.49999999682364393, 0.4296856430544671, 0.48790405860688757, 0.49999999682850876, 0.4290569323466237, 0.48937531017641744, 0.49999999680765145, 0.4282771037327572, 0.49141089829400086, 0.49999999679969076, 0.4276055216341146, 0.49412073741268026, 0.4999999967812793, 0.4275047468374559, 0.4946098814384686, 0.49999999668854406, 0.4263623157525364, 0.48258837329156884, 0.2500391219820913, 0.2500261108782112, 0.25001499226273